# L09 · Assignment — Recipe semantic search

> *Sarah's product search demo went well. Marcus's friend who runs a recipe website asks her for the same thing — but for recipes. The corpus is small (30 recipes) but the queries are even more synonym-heavy: "creamy Italian dessert", "spicy something with rice", "summery starter".*

In this assignment you will:

**Part A.** Build a semantic search engine on a recipe corpus. Apply it to a benchmark of 6 natural-language queries.

**Part B.** Build a TF-IDF baseline and compare. **Target: semantic ≥ TF-IDF on top-1.**

**Part C.** Reflect on **two** specific failure modes and how to fix them.

Total runtime: ~5 minutes.

---

## Setup

In [ ]:
# --- Colab setup: install sentence-transformers if missing (no-op in the local dsai-m3 env) ---
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)

import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

torch.set_num_threads(1)
np.random.seed(0)

# Load data — a fresh corpus of 30 recipes (instead of products), with even more
# synonym-heavy queries. Everything you learned on the shop catalogue applies here.
# Use the local file if present (VS Code), else fetch from GitHub (works in Colab).
import os
_LOCAL = 'data/recipes.csv'
_URL = 'https://raw.githubusercontent.com/su-ntu-ctp/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/recipes.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)
print(f"Recipes: {len(df)}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
df.head(3)

---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Part A — build the semantic search engine + benchmark |
| **🔵 Advanced Track** | Learners with prior ML background | Run the shared Setup + Part A engine cells, then Part B (TF-IDF baseline) + Part C (failure-mode reflection) |

If you're unsure, start with the **Foundational Track**. Both tracks cover the same lesson outcomes; only the scaffolding differs. **Everyone runs the shared Setup cell and Part A's three engine cells** — they build the semantic search engine that Parts B and C compare against. The Advanced Track simply adds the TF-IDF baseline and the written reflection.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


---

## Part A — Semantic search

Load the pretrained model, embed every recipe, build a `search()` function.

### A.1 · Embed the corpus

In [ ]:
# Load the pretrained model — turns text into vectors
model = SentenceTransformer('all-MiniLM-L6-v2')

# Build the document text — name + description
documents = (df['name'] + ' — ' + df['description']).tolist()

print(f"Embedding {len(documents)} recipes...")
# model.encode() turns every recipe into a vector — the one-time indexing step
embeddings = model.encode(documents, show_progress_bar=False)
print(f"Embeddings shape: {embeddings.shape}")

### A.2 · Define the search function

In [ ]:
def semantic_search(query, top=5):
    q = model.encode([query])                      # embed the query the same way as the recipes
    sims = cosine_similarity(q, embeddings)[0]     # compare it to every recipe
    order = np.argsort(-sims)[:top]                # keep the closest matches
    return df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

# Sanity check — confirm 'creamy Italian dessert' returns sensible results
print("Query: 'creamy Italian dessert'")
print(semantic_search('creamy Italian dessert', top=3)[['category','name','score']].to_string(index=False))

### A.3 · The benchmark queries

Your engine will be evaluated against these 6 queries:

In [ ]:
# The benchmark — six natural-language queries, each with its expected winning recipe (the answer key)
BENCHMARK = {
    "creamy Italian dessert":          "Tiramisu",
    "spicy curry with coconut milk":   "Thai Green Curry",
    "refreshing fruit smoothie":       "Mango Smoothie",
    "Mediterranean salad with feta":   "Greek Village Salad",
    "rich chocolate pudding":          "Chocolate Lava Cake",
    "vegan main with beans":           "Vegan Chickpea Curry",
}

correct = 0
print(f"{'Query':40s} {'Top-1 prediction':30s} {'Expected':25s}")
print('-' * 100)
# Score your engine's top-1 prediction against the answer key
for q, expected in BENCHMARK.items():
    top1 = semantic_search(q, top=1).iloc[0]['name']
    ok = top1 == expected
    correct += int(ok)
    print(f"  {'✅' if ok else '❌'}  {q:36s} {top1:30s} {expected:25s}")
sem_acc = correct / len(BENCHMARK)
print(f"\n[Part A] Semantic top-1 accuracy: {correct}/{len(BENCHMARK)} = {sem_acc:.0%}")

---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. First run the shared Setup cell and Part A's engine cells above (they define `model`, `documents`, and `semantic_search`); then complete Parts B and C below.*

---


---

## Part B — TF-IDF baseline

Build a TF-IDF retriever on the same corpus.

In [ ]:
# BENCHMARK is defined in Part A; redefined here so the Advanced Track runs standalone.
BENCHMARK = {
    "creamy Italian dessert":          "Tiramisu",
    "spicy curry with coconut milk":   "Thai Green Curry",
    "refreshing fruit smoothie":       "Mango Smoothie",
    "Mediterranean salad with feta":   "Greek Village Salad",
    "rich chocolate pudding":          "Chocolate Lava Cake",
    "vegan main with beans":           "Vegan Chickpea Curry",
}

# Build the classical keyword-based retriever on the same recipes — the baseline to compare against
vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vec.fit_transform(documents)
print(f"TF-IDF vocab: {len(vec.vocabulary_)}, matrix shape: {tfidf_matrix.shape}")

def tfidf_search(query, top=5):
    q = vec.transform([query])                       # vectorise the query in the same TF-IDF space
    sims = cosine_similarity(q, tfidf_matrix)[0]
    order = np.argsort(-sims)[:top]
    return df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

# Run the same benchmark as Part A, so the two approaches are directly comparable
correct = 0
print(f"{'Query':40s} {'TF-IDF top-1':30s} {'Expected':25s}")
print('-' * 100)
for q, expected in BENCHMARK.items():
    top1 = tfidf_search(q, top=1).iloc[0]['name']
    ok = top1 == expected
    correct += int(ok)
    print(f"  {'✅' if ok else '❌'}  {q:36s} {top1:30s} {expected:25s}")
tf_acc = correct / len(BENCHMARK)
print(f"\n[Part B] TF-IDF top-1 accuracy: {correct}/{len(BENCHMARK)} = {tf_acc:.0%}")

### B.1 · Head-to-head

In [ ]:
# Print both accuracies side by side
print(f"{'Approach':12s} {'Top-1 acc':>10s}")
print('-' * 24)
print(f"{'Semantic':12s} {sem_acc:>10.0%}")
print(f"{'TF-IDF':12s} {tf_acc:>10.0%}")
print()
# The target you must hit: semantic ≥ TF-IDF on top-1
print(f"Target: Semantic ≥ TF-IDF: {'✅ PASS' if sem_acc >= tf_acc else '❌ FAIL — investigate'}")

---

## Part C — Reflect on failure modes

Identify **two** queries from the benchmark where semantic search failed (or where its top-1 wasn't ideal), and explain:

1. **Why** did the model rank the wrong recipe first?
2. **What** could you do to fix it in a production system?

Semantic search often gets all six benchmark queries right, so two good sources of failure material are: (a) the **two least confident** benchmark queries (lowest top-1 score), and (b) the **stretch queries** in C.1b below, which are designed to break or confuse the model.

### C.1 · Identify candidates for failure

In [ ]:
# For each query, show top-3 results and their scores — use this to spot where the model is
# uncertain or wrong, then write your reflection on WHY it happened and WHAT production signal
# (a filter, a re-ranker, more context) would fix it.
for q, expected in BENCHMARK.items():
    print(f"\nQuery: {q!r}  (expected: {expected})")
    top3 = semantic_search(q, top=3)
    print(top3[['name','category','score']].to_string(index=False))

### C.1b · Stretch queries — deliberately hard

The six benchmark queries are fairly *easy*: the right recipe's name usually shares a word with the query, so semantic search tends to nail them. Real customers aren't that tidy.

The three **stretch queries** below are intentionally hard — they use paraphrase, metaphor, or are genuinely ambiguous, so there is **no single correct answer**. Run the cell, read the top-3 for each, and decide for yourself whether the result is defensible. These are your best material for the reflection in C.2.

In [ ]:
# Three HARD searches with no clean answer key — inspect the top-3 and judge whether the
# model's pick is reasonable. This is exactly the kind of failure analysis production teams do.
STRETCH_QUERIES = [
    ("a comforting dish for a cold winter night", "vague intent -- lasagna, crumble, or a curry could all fit"),
    ("an impressive dessert to end a dinner party", "metaphor -- 'impressive' appears in no description"),
    ("a refreshing non-alcoholic drink", "ambiguous -- could be the smoothie or the matcha latte"),
]

for q, why in STRETCH_QUERIES:
    print(f"\nQuery: {q!r}")
    print(f"  (why it's hard: {why})")
    top3 = semantic_search(q, top=3)
    print(top3[['name', 'category', 'score']].to_string(index=False))

### C.2 · Your reflection

Pick **two** queries — from the benchmark or (better) the stretch queries in C.1b. For each, write 2-3 sentences explaining:

- WHY the model behaved as it did
- WHAT signal (filter, re-ranker, additional context) would fix it in production

**Query 1 reflection:** *(your answer)*

**Query 2 reflection:** *(your answer)*

### C.3 · Open-ended improvement

The benchmark queries are easy; the stretch queries show how quickly accuracy drops on realistic, messy language. Imagine a larger, noisier corpus where semantic search reaches only ~80% top-1: if you had to add ONE thing to push it from 80% to 95%, what would it be? (one sentence)

*(your answer)*

---

## Submission checklist

- [ ] Part A — Semantic search built, benchmarked, accuracy printed
- [ ] Part B — TF-IDF baseline built, head-to-head comparison printed
- [ ] Part B target — Semantic ≥ TF-IDF (PASS)
- [ ] Part C — Two failure-mode reflections completed
- [ ] Part C — One open-ended improvement suggestion

Save the notebook with outputs and submit.